In [1]:
import os
import xml.etree.ElementTree as ET
from lxml import etree 
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import warnings

In [2]:
ns = { "tei" : "http://www.tei-c.org/ns/1.0" }
parser = etree.XMLParser(recover=True)

In [3]:
warnings.filterwarnings("ignore")

In [10]:
#Choose name of book and category (for now: either 'graphic', 'table' or 'music')
# Other categories can be added, remember to create the corresponding folder in 'music_examples'
def extract_images(book, category):
    tree = ET.parse(f'../tei-xml/{book}.xml', parser=parser)
    root = tree.getroot()
    
    for facsimile in root.findall(".//tei:facsimile", ns):
        img_url = facsimile.find(".//tei:graphic", ns).attrib["url"]
        
        if category=='music':
            rendition = 'Music'
        elif category=='graphic':
            rendition = 'Graphic'
        elif category=='table':
            rendition = 'Table'
        else:
            print('invalid category: choose music, table or graphic')
            break
        for zone in facsimile.findall(f".//tei:zone[@rendition='{rendition}']", ns):
            img_id = zone.attrib["{http://www.w3.org/XML/1998/namespace}id"]
            
            points = zone.attrib["points"]
            points = points.split()
            points = [list(map(int, p.split(','))) for p in points]        
            points = np.array(points)
            x1, y1 = points.min(axis=0)
            x2, y2 = points.max(axis=0)

            img_path = f'../scans/{book}/' + img_url 
            
            img = Image.open(img_path)
            
            music_region = img.crop((x1,y1,x2,y2))
            music_region.save(f'../extracted_images/{book}/{category}/{img_id}.png')


In [13]:
#Extract images for the whole list of books
for filename in tqdm([f[:-4] for f in os.listdir('../tei-xml') if f.endswith('.xml')]):
    extract_images(filename, category='graphic')

100%|██████████| 12/12 [03:17<00:00, 16.45s/it]
